In [4]:
%pip install transformers datasets soundfile speechbrain accelerate

  Using cached ruamel.yaml-0.18.15-py3-none-any.whl.metadata (25 kB)
   ---------------------------------------- 0.0/864.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/864.1 kB ? eta -:--:--
   ------------ --------------------------- 262.1/864.1 kB ? eta -:--:--
   ---------------------------------------- 864.1/864.1 kB 5.5 MB/s  0:00:00
Using cached ruamel.yaml-0.18.15-py3-none-any.whl (119 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ----------------------------- ---------- 0.8/1.1 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 3.9 MB/s  0:00:00

  Attempting uninstall: ruamel.yaml

    Found existing installation: ruamel.yaml 0.17.21

    Uninstalling ruamel.yaml-0.17.21:

   ------------- -------------------------- 2/6 [ruamel.yaml]
   ------------- -------------------------- 2/6 [ruamel.yaml]
   ------------- -------------------------- 2/6 [ruamel.yaml]
   ------------- -------------------------- 2

# Loading dataset

In [6]:
from datasets import load_dataset, Audio

dataset = load_dataset("facebook/voxpopuli", "nl", split="train", trust_remote_code=True)
len(dataset)

n_files.json: 0.00B [00:00, ?B/s]

data/nl/asr_train.tsv:   0%|          | 0.00/7.09M [00:00<?, ?B/s]

data/nl/asr_dev.tsv:   0%|          | 0.00/422k [00:00<?, ?B/s]

data/nl/asr_test.tsv:   0%|          | 0.00/405k [00:00<?, ?B/s]

data/nl/train/train_part_0.tar.gz:   0%|          | 0.00/1.31G [00:00<?, ?B/s]

...

data/nl/train/train_part_3.tar.gz:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

data/nl/train/train_part_4.tar.gz:   0%|          | 0.00/255M [00:00<?, ?B/s]

data/nl/dev/dev_part_0.tar.gz:   0%|          | 0.00/321M [00:00<?, ?B/s]

data/nl/test/test_part_0.tar.gz:   0%|          | 0.00/319M [00:00<?, ?B/s]

OSError: [Errno 22] Invalid argument: 'C:\\Users\\natha\\.cache\\huggingface\\datasets\\downloads\\extracted\\6bef4c3147bec32b63ea059ed16a012314182d8ad7c688d0b120f9335aa53d79\\train_part_0\\20100210-0900-PLENARY-3-nl_20100210-09:06:43_4.wav'

In [7]:
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

NameError: name 'dataset' is not defined

# Preprocessing

In [ ]:
from transformers import SpeechT5Processor

checkpoint = "microsoft/speecht5_tts"
processor = SpeechT5Processor.from_pretrained(checkpoint)

In [ ]:
tokenizer = processor.tokenizer

In [ ]:
dataset[0]

In [ ]:
def extract_all_chars(batch):
    all_text = " ".join(batch["normalized_text"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}


vocabs = dataset.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=dataset.column_names,
)

dataset_vocab = set(vocabs["vocab"][0])
tokenizer_vocab = {k for k, _ in tokenizer.get_vocab().items()}

dataset_vocab - tokenizer_vocab

In [ ]:
replacements = [
    ("à", "a"),
    ("ç", "c"),
    ("è", "e"),
    ("ë", "e"),
    ("í", "i"),
    ("ï", "i"),
    ("ö", "o"),
    ("ü", "u"),
]


def cleanup_text(inputs):
    for src, dst in replacements:
        inputs["normalized_text"] = inputs["normalized_text"].replace(src, dst)
    return inputs


dataset = dataset.map(cleanup_text)

# Speaker

In [ ]:
from collections import defaultdict

speaker_counts = defaultdict(int)

for speaker_id in dataset["speaker_id"]:
    speaker_counts[speaker_id] += 1

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(speaker_counts.values(), bins=20)
plt.ylabel("Speakers")
plt.xlabel("Examples")
plt.show()

In [ ]:
def select_speaker(speaker_id):
    return 100 <= speaker_counts[speaker_id] <= 400


dataset = dataset.filter(select_speaker, input_columns=["speaker_id"])

len(set(dataset["speaker_id"]))

In [ ]:
len(dataset)

# Embedding

In [ ]:
import os
import torch
from speechbrain.pretrained import EncoderClassifier

spk_model_name = "speechbrain/spkrec-xvect-voxceleb"

device = "cuda" if torch.cuda.is_available() else "cpu"
speaker_model = EncoderClassifier.from_hparams(
    source=spk_model_name,
    run_opts={"device": device},
    savedir=os.path.join("/tmp", spk_model_name),
)


def create_speaker_embedding(waveform):
    with torch.no_grad():
        speaker_embeddings = speaker_model.encode_batch(torch.tensor(waveform))
        speaker_embeddings = torch.nn.functional.normalize(speaker_embeddings, dim=2)
        speaker_embeddings = speaker_embeddings.squeeze().cpu().numpy()
    return speaker_embeddings

# Process the dataset

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]

    example = processor(
        text=example["normalized_text"],
        audio_target=audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_attention_mask=False,
    )

    # strip off the batch dimension
    example["labels"] = example["labels"][0]

    # use SpeechBrain to obtain x-vector
    example["speaker_embeddings"] = create_speaker_embedding(audio["array"])

    return example

processed_example = prepare_dataset(dataset[0])
list(processed_example.keys())

In [ ]:
processed_example["speaker_embeddings"].shape

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.imshow(processed_example["labels"].T)
plt.show()